# Silver population place reconciliation

## Purpose

Create one explainable reconciliation result for every current Census Table C source row. Each row keeps its file, sheet, row number, raw place name, raw population, selected Bronze snapshot, and Bronze ingest run. Matching uses verified PSGC context first, active approved aliases second, and deterministic name normalization last. Ambiguous, unmatched, and invalid rows remain visible. The natural key is source file, sheet, row number, and deterministic Silver run ID.


## 1. Set the version contract and safety gates

The source-supported population reference date is July 1, 2024. A single active alias version may participate. An empty alias table is valid and becomes `NO_APPROVED_ALIAS`. The latest Table C load must be successful and its latest blocking Bronze validation must pass. The PSGC Silver output must match the latest PSGC Bronze snapshot.


In [0]:
USE CATALOG `buildabida-capstone`;

CREATE OR REPLACE TEMPORARY VIEW population_pipeline_parameters AS
SELECT
    DATE'2024-07-01' AS population_reference_date,
    2024 AS reference_year,
    'PSA 2024 Census Table C' AS source_name,
    'silver-population-place-v2' AS transformation_rule_version,
    COALESCE(MAX(alias_version), 'NO_APPROVED_ALIAS') AS alias_version,
    COUNT(DISTINCT alias_version) AS active_alias_version_count
FROM `02-silver`.config_place_name_alias
WHERE source_system = 'census_2024_table_c'
  AND approval_status = 'APPROVED'
  AND is_active = TRUE;

SELECT ASSERT_TRUE(
    active_alias_version_count <= 1,
    'Population matching requires at most one active approved Table C alias version'
)
FROM population_pipeline_parameters;

CREATE OR REPLACE TEMPORARY VIEW latest_population_load AS
SELECT * EXCEPT (terminal_rank)
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY table_name
            ORDER BY COALESCE(completed_at, started_at) DESC, started_at DESC
        ) AS terminal_rank
    FROM `01-bronze`.load_log
    WHERE table_name = 'census_2024_table_c'
      AND status IN ('STARTED', 'SUCCESS', 'FAILED', 'SKIPPED_IDEMPOTENT')
)
WHERE terminal_rank = 1;

SELECT ASSERT_TRUE(
    COUNT(*) = 1
    AND COUNT_IF(status IN ('SUCCESS', 'SKIPPED_IDEMPOTENT')) = 1
    AND COUNT_IF(snapshot_id IS NOT NULL AND TRIM(snapshot_id) <> '') = 1
    AND COUNT_IF(rows_loaded > 0) = 1,
    'Table C Bronze safety gate failed: latest audit must be successful, identified, and non-empty'
)
FROM latest_population_load;

CREATE OR REPLACE TEMPORARY VIEW latest_population_validation AS
SELECT * EXCEPT (validation_rank)
FROM (
    SELECT
        *,
        DENSE_RANK() OVER (ORDER BY run_ts DESC, run_id DESC) AS validation_rank
    FROM `04-validation`.dq_results
    WHERE table_name = 'census_2024_table_c'
      AND snapshot_id = (SELECT snapshot_id FROM latest_population_load)
)
WHERE validation_rank = 1;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT_IF(action = 'stop' AND status IN ('FAIL', 'ERROR')) = 0,
    'Table C Bronze safety gate failed: latest blocking validation did not pass'
)
FROM latest_population_validation;

CREATE OR REPLACE TEMPORARY VIEW latest_psgc_load_for_population AS
SELECT * EXCEPT (terminal_rank)
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY table_name
            ORDER BY COALESCE(completed_at, started_at) DESC, started_at DESC
        ) AS terminal_rank
    FROM `01-bronze`.load_log
    WHERE table_name = 'psgc'
      AND status IN ('STARTED', 'SUCCESS', 'FAILED', 'SKIPPED_IDEMPOTENT')
)
WHERE terminal_rank = 1;

SELECT ASSERT_TRUE(
    COUNT(*) = 1
    AND COUNT_IF(status IN ('SUCCESS', 'SKIPPED_IDEMPOTENT')) = 1
    AND COUNT_IF(snapshot_id IS NOT NULL AND TRIM(snapshot_id) <> '') = 1
    AND COUNT_IF(rows_loaded > 0) = 1,
    'PSGC Bronze safety gate failed before population matching'
)
FROM latest_psgc_load_for_population;

SELECT ASSERT_TRUE(
    COUNT(DISTINCT source_snapshot_id) = 1
    AND MAX(source_snapshot_id) = (SELECT snapshot_id FROM latest_psgc_load_for_population),
    'silver_psgc_place is not aligned with the latest safe PSGC Bronze snapshot'
)
FROM `02-silver`.silver_psgc_place;


## 2. Preserve Table C rows and infer verified context

Table C provides file, sheet, row, place name, and population, but no reliable PSGC code. The pipeline safely parses population and retains its raw value.

Sheet names establish province, independent-city, or Special Geographic Area context only when they resolve to one PSGC place. The exact name is tried first. When it finds nothing, one approved sheet alias that names the target's region may resolve the sheet, such as `City of Lapu` and `SGA`.

Locality headings establish the current city, municipality, or Manila district only when they resolve uniquely inside that sheet context. The exact name is tried first, then an approved heading alias with province or region context, then the shared normalization key. A Manila district or SGA municipality belongs to its sheet through the verified PSGC place at its five-digit code, because neither has a `province_code`. This context is carried to later barangay rows in source-row order.


In [0]:
CREATE OR REPLACE TEMPORARY VIEW table_c_source AS
SELECT
    source_file,
    sheet_name,
    source_row_number,
    place_name AS place_name_raw,
    population AS population_raw,
    TRY_CAST(REPLACE(TRIM(population), ',', '') AS BIGINT) AS population_count,
    REGEXP_REPLACE(
        REGEXP_REPLACE(UPPER(TRIM(place_name)), '\\s+', ' '),
        '\\s*\\*+$',
        ''
    ) AS place_name_standardized,
    REGEXP_REPLACE(
        REGEXP_REPLACE(
            REGEXP_REPLACE(UPPER(TRIM(place_name)), '\\s+', ' '),
            '\\s*\\*+$',
            ''
        ),
        '\\s+[0-9]+$',
        ''
    ) AS heading_name_standardized,
    NULLIF(
        REGEXP_REPLACE(
            REGEXP_REPLACE(
                REGEXP_REPLACE(UPPER(TRIM(place_name)), '\\s+', ' '),
                '\\s*\\*+$',
                ''
            ),
            '\\s*[0-9]+$',
            ''
        ),
        ''
    ) AS place_name_without_footnote,
    REGEXP_REPLACE(UPPER(TRIM(sheet_name)), '\\s+', ' ') AS sheet_name_standardized,
    REGEXP_REPLACE(TRIM(place_name), '\\s+', ' ') = UPPER(REGEXP_REPLACE(TRIM(place_name), '\\s+', ' '))
        AS is_uppercase_heading,
    MIN(TRY_CAST(source_row_number AS BIGINT)) OVER (PARTITION BY source_file, sheet_name)
        AS sheet_first_row_number,
    _source_snapshot_id AS source_snapshot_id,
    _ingest_run_id AS source_ingest_run_id,
    _ingested_at AS source_load_ts,
    SHA2(CONCAT_WS('|', _source_snapshot_id, source_file, sheet_name, source_row_number), 256)
        AS source_row_identity
FROM `01-bronze`.census_2024_table_c;

-- One normalization key shared by Table C and PSGC names, so both sides follow identical rules.
-- It removes superscript footnote digits and a trailing '*', reads STO./STA./ST. as SANTO/SANTA/SAINT,
-- and treats a hyphen, a space between letters, and no separator as the same. It never scores similarity.
CREATE OR REPLACE TEMPORARY VIEW place_name_match_key AS
WITH names AS (
    SELECT EXPLODE(ARRAY(place_name_standardized, heading_name_standardized, place_name_without_footnote))
        AS name_standardized
    FROM table_c_source
    UNION
    SELECT place_name_standardized FROM `02-silver`.silver_psgc_place
),
without_marks AS (
    SELECT
        name_standardized,
        REGEXP_REPLACE(
            TRIM(REGEXP_REPLACE(REGEXP_REPLACE(name_standardized, '[⁰¹²³⁴⁵⁶⁷⁸⁹]', ''), '\\s+', ' ')),
            '\\s*\\*+$',
            ''
        ) AS name_without_marks
    FROM names
    WHERE name_standardized IS NOT NULL AND name_standardized <> ''
),
-- Padding the name with spaces lets each pattern match a whole word, so the rule needs no
-- backreference or lookahead and behaves the same on every SQL engine.
with_saint_words AS (
    SELECT
        name_standardized,
        TRIM(
            REGEXP_REPLACE(
                REGEXP_REPLACE(
                    REGEXP_REPLACE(CONCAT(' ', name_without_marks, ' '), ' STO\\.? ', ' SANTO '),
                    ' STA\\.? ',
                    ' SANTA '
                ),
                ' ST\\.? ',
                ' SAINT '
            )
        ) AS name_with_saint_words
    FROM without_marks
)
SELECT
    name_standardized,
    REGEXP_REPLACE(REPLACE(name_with_saint_words, '-', ''), '(?<=\\p{L}) (?=\\p{L})', '') AS name_match_key
FROM with_saint_words;

-- PSGC places with their match key and the verified place at their five-digit (province-level) code.
-- For most localities that parent is the province. It is the City of Manila for its districts and
-- the Special Geographic Area for its municipalities, neither of which has a province_code.
CREATE OR REPLACE TEMPORARY VIEW psgc_place_for_matching AS
SELECT
    place.*,
    name_key.name_match_key,
    CASE
        WHEN place.place_type IN ('CITY', 'MUNICIPALITY', 'SUBMUNICIPALITY')
         AND parent.psgc_code <> place.psgc_code
            THEN parent.psgc_code
    END AS locality_parent_code
FROM `02-silver`.silver_psgc_place AS place
LEFT JOIN place_name_match_key AS name_key
    ON place.place_name_standardized = name_key.name_standardized
LEFT JOIN `02-silver`.silver_psgc_place AS parent
    ON parent.psgc_code = CONCAT(SUBSTRING(place.psgc_code, 1, 5), '00000')
   AND parent.source_snapshot_id = place.source_snapshot_id;

CREATE OR REPLACE TEMPORARY VIEW table_c_sheet_exact_candidates AS
SELECT
    source.source_file,
    source.sheet_name,
    CASE
        WHEN COUNT(DISTINCT CASE WHEN place.place_type IN ('PROVINCE', 'CITY') THEN place.psgc_code END) > 0
            THEN COUNT(DISTINCT CASE WHEN place.place_type IN ('PROVINCE', 'CITY') THEN place.psgc_code END)
        ELSE COUNT(DISTINCT CASE WHEN place.place_type = 'MUNICIPALITY' THEN place.psgc_code END)
    END AS candidate_count,
    CASE
        WHEN COUNT(DISTINCT CASE WHEN place.place_type IN ('PROVINCE', 'CITY') THEN place.psgc_code END) = 1
            THEN MAX(CASE WHEN place.place_type IN ('PROVINCE', 'CITY') THEN place.psgc_code END)
        WHEN COUNT(DISTINCT CASE WHEN place.place_type IN ('PROVINCE', 'CITY') THEN place.psgc_code END) = 0
         AND COUNT(DISTINCT CASE WHEN place.place_type = 'MUNICIPALITY' THEN place.psgc_code END) = 1
            THEN MAX(CASE WHEN place.place_type = 'MUNICIPALITY' THEN place.psgc_code END)
    END AS candidate_psgc_code
FROM (
    SELECT DISTINCT source_file, sheet_name, sheet_name_standardized
    FROM table_c_source
) AS source
LEFT JOIN `02-silver`.silver_psgc_place AS place
    ON source.sheet_name_standardized = place.place_name_standardized
   AND place.place_type IN ('PROVINCE', 'CITY', 'MUNICIPALITY')
GROUP BY source.source_file, source.sheet_name;

-- A sheet whose name matches no PSGC place may use an approved sheet alias. The alias must name the
-- target's region, so a blank context never authorizes a match anywhere.
CREATE OR REPLACE TEMPORARY VIEW table_c_sheet_alias_candidates AS
SELECT
    source.source_file,
    source.sheet_name,
    COUNT(DISTINCT target.psgc_code) AS candidate_count,
    MAX(target.psgc_code) AS candidate_psgc_code
FROM (
    SELECT DISTINCT source_file, sheet_name, sheet_name_standardized
    FROM table_c_source
) AS source
CROSS JOIN population_pipeline_parameters AS parameters
JOIN `02-silver`.config_place_name_alias AS alias
    ON alias.source_system = 'census_2024_table_c'
   AND alias.alias_version = parameters.alias_version
   AND alias.approval_status = 'APPROVED'
   AND alias.is_active = TRUE
   AND alias.match_rule = 'TABLE_C_SHEET_NAME'
   AND TRIM(alias.raw_region_name) <> ''
   AND REGEXP_REPLACE(UPPER(TRIM(alias.raw_place_name)), '\\s+', ' ') = source.sheet_name_standardized
JOIN `02-silver`.silver_psgc_place AS target
    ON alias.canonical_psgc_code = target.psgc_code
   AND UPPER(alias.place_type) = target.place_type
   AND target.place_type IN ('PROVINCE', 'CITY', 'MUNICIPALITY', 'UNASSIGNED')
   AND REGEXP_REPLACE(UPPER(TRIM(alias.raw_region_name)), '\\s+', ' ') = REGEXP_REPLACE(UPPER(TRIM(target.region_name)), '\\s+', ' ')
   AND (alias.psgc_version IS NULL OR alias.psgc_version = target.psgc_version)
GROUP BY source.source_file, source.sheet_name;

CREATE OR REPLACE TEMPORARY VIEW table_c_sheet_candidates AS
SELECT
    exact.source_file,
    exact.sheet_name,
    CASE WHEN exact.candidate_count > 0 THEN exact.candidate_count ELSE COALESCE(alias.candidate_count, 0) END
        AS candidate_count,
    CASE WHEN exact.candidate_count > 0 THEN exact.candidate_psgc_code ELSE alias.candidate_psgc_code END
        AS candidate_psgc_code,
    CASE
        WHEN exact.candidate_count > 0 THEN 'EXACT_NAME'
        WHEN alias.candidate_count > 0 THEN 'APPROVED_ALIAS'
        ELSE 'NONE'
    END AS sheet_context_method
FROM table_c_sheet_exact_candidates AS exact
LEFT JOIN table_c_sheet_alias_candidates AS alias
    ON exact.source_file = alias.source_file
   AND exact.sheet_name = alias.sheet_name;

CREATE OR REPLACE TEMPORARY VIEW table_c_with_sheet_context AS
SELECT
    source.*,
    CASE WHEN sheet.candidate_count = 1 THEN sheet.candidate_psgc_code END AS sheet_psgc_code,
    sheet.candidate_count AS sheet_candidate_count,
    sheet.sheet_context_method,
    context.place_type AS sheet_place_type,
    context.region_code AS sheet_region_code,
    context.region_name AS sheet_region_name,
    context.province_code AS sheet_province_code,
    context.province_name AS sheet_province_name
FROM table_c_source AS source
LEFT JOIN table_c_sheet_candidates AS sheet
    ON source.source_file = sheet.source_file
   AND source.sheet_name = sheet.sheet_name
LEFT JOIN `02-silver`.silver_psgc_place AS context
    ON sheet.candidate_count = 1
   AND sheet.candidate_psgc_code = context.psgc_code;

-- Cities, municipalities, and districts inside each verified sheet place. A district or SGA
-- municipality belongs to the sheet through locality_parent_code.
CREATE OR REPLACE TEMPORARY VIEW table_c_sheet_localities AS
SELECT
    sheet.source_file,
    sheet.sheet_name,
    place.psgc_code,
    place.place_type,
    place.place_name_standardized,
    place.name_match_key,
    place.psgc_version
FROM (
    SELECT DISTINCT source_file, sheet_name, sheet_psgc_code, sheet_place_type
    FROM table_c_with_sheet_context
    WHERE sheet_psgc_code IS NOT NULL
) AS sheet
JOIN psgc_place_for_matching AS place
    ON place.place_type IN ('CITY', 'MUNICIPALITY', 'SUBMUNICIPALITY')
   AND (
       place.province_code = sheet.sheet_psgc_code
       OR place.locality_parent_code = sheet.sheet_psgc_code
       OR (sheet.sheet_place_type IN ('CITY', 'MUNICIPALITY') AND place.psgc_code = sheet.sheet_psgc_code)
   );

-- Heading waterfall: exact name (rank 1), then an approved heading alias (rank 2), then the shared
-- normalization key (rank 3). A later rank is used only when every earlier rank found nothing.
CREATE OR REPLACE TEMPORARY VIEW table_c_locality_heading_candidates AS
WITH heading_rows AS (
    SELECT source.*, name_key.name_match_key AS heading_match_key
    FROM table_c_with_sheet_context AS source
    LEFT JOIN place_name_match_key AS name_key
        ON source.heading_name_standardized = name_key.name_standardized
    WHERE TRY_CAST(source.source_row_number AS BIGINT) <> source.sheet_first_row_number
)
SELECT
    source.source_row_identity,
    CASE WHEN source.heading_name_standardized = locality.place_name_standardized THEN 1 ELSE 3 END AS match_rank,
    CASE
        WHEN source.heading_name_standardized = locality.place_name_standardized THEN 'EXACT_CONTEXT'
        ELSE 'NORMALIZED_NAME_CONTEXT'
    END AS match_method,
    locality.psgc_code
FROM heading_rows AS source
JOIN table_c_sheet_localities AS locality
    ON source.source_file = locality.source_file
   AND source.sheet_name = locality.sheet_name
   AND (source.is_uppercase_heading OR locality.place_type = 'SUBMUNICIPALITY')
   AND (
       source.heading_name_standardized = locality.place_name_standardized
       OR source.heading_match_key = locality.name_match_key
   )
UNION ALL
SELECT source.source_row_identity, 2, 'APPROVED_ALIAS', locality.psgc_code
FROM heading_rows AS source
CROSS JOIN population_pipeline_parameters AS parameters
JOIN `02-silver`.config_place_name_alias AS alias
    ON alias.source_system = 'census_2024_table_c'
   AND alias.alias_version = parameters.alias_version
   AND alias.approval_status = 'APPROVED'
   AND alias.is_active = TRUE
   AND alias.match_rule = 'TABLE_C_LOCALITY_HEADING'
   AND REGEXP_REPLACE(UPPER(TRIM(alias.raw_place_name)), '\\s+', ' ') = source.heading_name_standardized
   AND (TRIM(alias.raw_region_name) <> '' OR TRIM(alias.raw_province_name) <> '')
   AND (TRIM(alias.raw_region_name) = '' OR REGEXP_REPLACE(UPPER(TRIM(alias.raw_region_name)), '\\s+', ' ') = REGEXP_REPLACE(UPPER(TRIM(source.sheet_region_name)), '\\s+', ' '))
   AND (TRIM(alias.raw_province_name) = '' OR REGEXP_REPLACE(UPPER(TRIM(alias.raw_province_name)), '\\s+', ' ') = REGEXP_REPLACE(UPPER(TRIM(source.sheet_province_name)), '\\s+', ' '))
JOIN table_c_sheet_localities AS locality
    ON source.source_file = locality.source_file
   AND source.sheet_name = locality.sheet_name
   AND alias.canonical_psgc_code = locality.psgc_code
   AND UPPER(alias.place_type) = locality.place_type
   AND (alias.psgc_version IS NULL OR alias.psgc_version = locality.psgc_version)
WHERE source.is_uppercase_heading;

CREATE OR REPLACE TEMPORARY VIEW table_c_locality_headings AS
WITH ranked AS (
    SELECT
        *,
        MIN(match_rank) OVER (PARTITION BY source_row_identity) AS best_rank
    FROM table_c_locality_heading_candidates
)
SELECT
    source_row_identity,
    MIN(match_rank) AS match_rank,
    MAX(match_method) AS match_method,
    COUNT(DISTINCT psgc_code) AS candidate_count,
    MAX(psgc_code) AS candidate_psgc_code
FROM ranked
WHERE match_rank = best_rank
GROUP BY source_row_identity;

-- An all-caps row that resolves to no heading may still be a barangay written in capitals, such as
-- H-2 or UEP I. It is tried as a barangay only when it is not the sheet subtotal and its name is not
-- any PSGC province, city, municipality, or district name, so an unresolved subtotal is never counted.
CREATE OR REPLACE TEMPORARY VIEW table_c_with_locality_context AS
SELECT
    source.*,
    COALESCE(heading.candidate_count, 0) AS heading_candidate_count,
    place_key.name_match_key AS place_name_match_key,
    CASE
        WHEN source.place_name_without_footnote <> source.place_name_standardized
            THEN footnote_key.name_match_key
    END AS footnote_name_match_key,
    COALESCE(
        LAST_VALUE(
            CASE WHEN heading.candidate_count = 1 THEN heading.candidate_psgc_code END,
            TRUE
        ) OVER (
            PARTITION BY source.source_file, source.sheet_name
            ORDER BY TRY_CAST(source.source_row_number AS BIGINT)
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ),
        CASE WHEN source.sheet_place_type IN ('CITY', 'MUNICIPALITY') THEN source.sheet_psgc_code END
    ) AS locality_context_code,
    COALESCE(
        NOT source.is_uppercase_heading
        OR (
            heading.source_row_identity IS NULL
            AND TRY_CAST(source.source_row_number AS BIGINT) <> source.sheet_first_row_number
            AND higher_level_name.name_standardized IS NULL
        ),
        FALSE
    ) AS is_barangay_candidate_row
FROM table_c_with_sheet_context AS source
LEFT JOIN table_c_locality_headings AS heading
    ON source.source_row_identity = heading.source_row_identity
LEFT JOIN place_name_match_key AS place_key
    ON source.place_name_standardized = place_key.name_standardized
LEFT JOIN place_name_match_key AS footnote_key
    ON source.place_name_without_footnote = footnote_key.name_standardized
LEFT JOIN (
    SELECT DISTINCT place_name_standardized AS name_standardized
    FROM `02-silver`.silver_psgc_place
    WHERE place_type <> 'BARANGAY'
) AS higher_level_name
    ON source.place_name_standardized = higher_level_name.name_standardized;


## 3. Apply the deterministic matching waterfall

Every candidate must use verified sheet or locality context. The waterfall has four ranks. A later rank runs only when every earlier rank found nothing for the row.

1. **Exact name.** A province or independent-city subtotal can match its unique sheet place. A locality heading can match its unique contextual locality. Other rows can match only a barangay under the verified current locality. An all-caps row that resolves to no heading, such as `H-2` or `UEP I`, is tried as a barangay only when it is not the sheet subtotal and its name is not any PSGC province, city, municipality, or district name.
2. **Approved alias.** One active `APPROVED` alias from the selected alias version, with nonblank geographic context.
3. **Normalized name.** The shared normalization key, inside the same locality. It removes superscript footnote digits and a trailing `*`, reads `STO.`, `STA.`, and `ST.` as `SANTO`, `SANTA`, and `SAINT`, and treats a hyphen, a space between letters, and no separator as the same.
4. **Footnote number removed.** The name without its last number, such as `Muzon South 2` or `Zapote 3 5`, compared by the same key inside the same locality. Real names such as `Barangay 1` or `Zapote 3` already match at rank 1, so they never reach this rank.

Ranks 3 and 4 never take a PSGC place that an exact or alias match already claimed. When two rows reach the same place through them, neither is accepted. Global name-only and fuzzy matching are not accepted methods.


In [0]:
-- Every candidate carries its waterfall rank:
-- 1 exact contextual name, 2 approved contextual alias, 3 normalized name in the same locality,
-- 4 name without its last footnote number in the same locality.
CREATE OR REPLACE TEMPORARY VIEW table_c_alias_candidates AS
SELECT
    source.source_row_identity,
    target.psgc_code
FROM table_c_with_locality_context AS source
CROSS JOIN population_pipeline_parameters AS parameters
LEFT JOIN `02-silver`.silver_psgc_place AS locality
    ON source.locality_context_code = locality.psgc_code
JOIN `02-silver`.config_place_name_alias AS alias
    ON alias.source_system = 'census_2024_table_c'
   AND alias.alias_version = parameters.alias_version
   AND alias.approval_status = 'APPROVED'
   AND alias.is_active = TRUE
   AND COALESCE(alias.match_rule, '') NOT IN ('TABLE_C_SHEET_NAME', 'TABLE_C_LOCALITY_HEADING')
   AND REGEXP_REPLACE(UPPER(TRIM(alias.raw_place_name)), '\\s+', ' ') = source.place_name_standardized
   AND (
       TRIM(alias.raw_region_name) <> ''
       OR TRIM(alias.raw_province_name) <> ''
       OR TRIM(alias.raw_city_municipality_name) <> ''
   )
   AND (TRIM(alias.raw_region_name) = '' OR REGEXP_REPLACE(UPPER(TRIM(alias.raw_region_name)), '\\s+', ' ') = REGEXP_REPLACE(UPPER(TRIM(source.sheet_region_name)), '\\s+', ' '))
   AND (TRIM(alias.raw_province_name) = '' OR REGEXP_REPLACE(UPPER(TRIM(alias.raw_province_name)), '\\s+', ' ') = REGEXP_REPLACE(UPPER(TRIM(source.sheet_province_name)), '\\s+', ' '))
   AND (TRIM(alias.raw_city_municipality_name) = '' OR REGEXP_REPLACE(UPPER(TRIM(alias.raw_city_municipality_name)), '\\s+', ' ') = locality.place_name_standardized)
JOIN `02-silver`.silver_psgc_place AS target
    ON alias.canonical_psgc_code = target.psgc_code
   AND UPPER(alias.place_type) = target.place_type
   AND (alias.psgc_version IS NULL OR alias.psgc_version = target.psgc_version);

-- Barangay candidates come from one join on the verified locality. The CASE keeps the strongest rule
-- that links the row to that PSGC barangay.
CREATE OR REPLACE TEMPORARY VIEW table_c_match_candidates AS
SELECT source.source_row_identity, 1 AS match_rank, 'EXACT_CONTEXT' AS match_method, place.psgc_code
FROM table_c_with_locality_context AS source
JOIN `02-silver`.silver_psgc_place AS place
    ON source.sheet_candidate_count = 1
   AND source.sheet_psgc_code = place.psgc_code
   AND TRY_CAST(source.source_row_number AS BIGINT) = source.sheet_first_row_number
   AND source.place_name_standardized = place.place_name_standardized
UNION ALL
SELECT heading.source_row_identity, heading.match_rank, heading.match_method, heading.candidate_psgc_code
FROM table_c_locality_headings AS heading
WHERE heading.candidate_count = 1
UNION ALL
SELECT
    source.source_row_identity,
    CASE
        WHEN source.is_barangay_candidate_row AND source.place_name_standardized = place.place_name_standardized THEN 1
        WHEN source.place_name_match_key = place.name_match_key THEN 3
        ELSE 4
    END,
    CASE
        WHEN source.is_barangay_candidate_row AND source.place_name_standardized = place.place_name_standardized
            THEN 'EXACT_CONTEXT'
        WHEN source.place_name_match_key = place.name_match_key THEN 'NORMALIZED_NAME_CONTEXT'
        ELSE 'FOOTNOTE_REMOVED_CONTEXT'
    END,
    place.psgc_code
FROM table_c_with_locality_context AS source
JOIN psgc_place_for_matching AS place
    ON source.locality_context_code = place.city_municipality_code
   AND place.place_type = 'BARANGAY'
   AND (
       (source.is_barangay_candidate_row AND source.place_name_standardized = place.place_name_standardized)
       OR (NOT source.is_uppercase_heading AND source.place_name_match_key = place.name_match_key)
       OR (NOT source.is_uppercase_heading AND source.footnote_name_match_key = place.name_match_key)
   )
UNION ALL
SELECT source_row_identity, 2, 'APPROVED_ALIAS', psgc_code
FROM table_c_alias_candidates;

-- Exact and alias results (ranks 1 and 2) are selected first, and their PSGC places are claimed.
-- Normalized and footnote fallbacks (ranks 3 and 4) apply only to rows with no exact or alias
-- candidate, and never to a claimed PSGC place. When two fallback rows reach the same PSGC place,
-- target_claim_count exceeds 1 and neither row is accepted.
CREATE OR REPLACE TEMPORARY VIEW table_c_match_selection AS
WITH primary_ranked AS (
    SELECT
        *,
        MIN(CASE WHEN match_rank <= 2 THEN match_rank END) OVER (PARTITION BY source_row_identity) AS primary_rank
    FROM table_c_match_candidates
),
primary_counted AS (
    SELECT
        *,
        SIZE(COLLECT_SET(CASE WHEN match_rank = primary_rank THEN psgc_code END)
            OVER (PARTITION BY source_row_identity)) AS primary_candidate_count
    FROM primary_ranked
),
claimed AS (
    SELECT
        *,
        MAX(CASE WHEN match_rank = primary_rank AND primary_candidate_count = 1 THEN 1 ELSE 0 END)
            OVER (PARTITION BY psgc_code) = 1 AS is_claimed_target
    FROM primary_counted
),
eligible AS (
    SELECT
        *,
        primary_rank IS NOT NULL AND match_rank = primary_rank
            OR primary_rank IS NULL AND match_rank >= 3 AND NOT is_claimed_target AS is_eligible
    FROM claimed
),
best AS (
    SELECT
        *,
        MIN(CASE WHEN is_eligible THEN match_rank END) OVER (PARTITION BY source_row_identity) AS best_rank
    FROM eligible
),
per_row AS (
    SELECT
        source_row_identity,
        MIN(match_rank) AS match_rank,
        MAX(match_method) AS match_method,
        COUNT(DISTINCT psgc_code) AS candidate_count,
        MAX(psgc_code) AS candidate_psgc_code,
        MIN(match_rank) >= 3 AS is_fallback
    FROM best
    WHERE is_eligible AND match_rank = best_rank
    GROUP BY source_row_identity
)
SELECT
    source_row_identity,
    match_rank,
    match_method,
    candidate_count,
    candidate_psgc_code,
    CASE
        WHEN is_fallback
            THEN COUNT(CASE WHEN candidate_count = 1 THEN 1 END) OVER (PARTITION BY is_fallback, candidate_psgc_code)
        ELSE 1
    END AS target_claim_count
FROM per_row;


## 4. Publish one mutually exclusive result per source row

A row is matched only when the first rank with candidates has exactly one, and a fallback target is not shared with another row. Rank 1 gives `MATCHED_EXACT_CONTEXT`, rank 2 gives `MATCHED_ALIAS`, and ranks 3 and 4 give `MATCHED_NORMALIZED_CONTEXT`. `match_method` names the rule, and `sheet_context_method` shows whether the sheet resolved by name or by alias. Multiple candidates or a shared fallback target remain `AMBIGUOUS`, and zero candidates remain `UNMATCHED`.

Population uses safe parsing and is never replaced by PSGC. The final disposition is mutually exclusive, so its counts must reconcile exactly to Table C rows. Only matched, positive barangay records become `ACCEPTED_PRIMARY` for regional aggregation. Matched province, city, municipality, and Manila district subtotals stay `MATCHED_NON_PRIMARY_LEVEL`.


In [0]:
CREATE OR REPLACE TABLE `02-silver`.silver_population_place_reconciliation
USING DELTA
COMMENT 'One explainable PSGC reconciliation result per current Table C source row'
AS
WITH resolved AS (
    SELECT
        source.*,
        COALESCE(selection.candidate_count, 0) AS selected_candidate_count,
        selection.match_method AS selected_match_method,
        COALESCE(selection.target_claim_count, 0) AS target_claim_count,
        CASE
            WHEN selection.candidate_count = 1 AND selection.target_claim_count = 1
                THEN selection.candidate_psgc_code
        END AS matched_psgc_code,
        CASE
            WHEN source.source_file IS NULL OR source.sheet_name IS NULL
              OR source.source_row_number IS NULL OR NULLIF(TRIM(source.place_name_raw), '') IS NULL
                THEN 'INVALID_SOURCE'
            WHEN selection.candidate_count = 1 AND selection.target_claim_count = 1 THEN
                CASE selection.match_method
                    WHEN 'EXACT_CONTEXT' THEN 'MATCHED_EXACT_CONTEXT'
                    WHEN 'APPROVED_ALIAS' THEN 'MATCHED_ALIAS'
                    ELSE 'MATCHED_NORMALIZED_CONTEXT'
                END
            WHEN selection.candidate_count > 1 OR selection.target_claim_count > 1 THEN 'AMBIGUOUS'
            ELSE 'UNMATCHED'
        END AS match_status,
        parameters.* EXCEPT (active_alias_version_count)
    FROM table_c_with_locality_context AS source
    CROSS JOIN population_pipeline_parameters AS parameters
    LEFT JOIN table_c_match_selection AS selection
        ON source.source_row_identity = selection.source_row_identity
),
enriched AS (
    SELECT
        resolved.*,
        target.place_name_raw AS matched_place_name,
        target.place_type AS matched_place_type,
        target.region_code AS matched_region_code,
        target.region_name AS matched_region_name,
        target.province_code AS matched_province_code,
        target.city_municipality_code AS matched_city_municipality_code,
        target.psgc_population_crosscheck,
        target.psgc_version,
        target.source_snapshot_id AS psgc_source_snapshot_id,
        target.source_ingest_run_id AS psgc_source_ingest_run_id
    FROM resolved
    LEFT JOIN `02-silver`.silver_psgc_place AS target
        ON resolved.matched_psgc_code = target.psgc_code
)
SELECT
    source_file,
    sheet_name,
    source_row_number,
    source_row_identity,
    place_name_raw,
    place_name_standardized,
    population_raw,
    population_count,
    sheet_context_method,
    locality_context_code,
    matched_psgc_code,
    matched_place_name,
    matched_place_type,
    matched_region_code,
    matched_region_name,
    matched_province_code,
    matched_city_municipality_code,
    match_status,
    CASE
        WHEN match_status IN ('MATCHED_EXACT_CONTEXT', 'MATCHED_ALIAS', 'MATCHED_NORMALIZED_CONTEXT')
            THEN selected_match_method
        ELSE 'NO_ACCEPTED_METHOD'
    END AS match_method,
    CASE
        WHEN match_status = 'MATCHED_EXACT_CONTEXT' THEN 'HIGH'
        WHEN match_status = 'MATCHED_ALIAS' THEN 'REVIEWED'
        WHEN match_status = 'MATCHED_NORMALIZED_CONTEXT' THEN 'RULE_NORMALIZED'
        ELSE 'UNRESOLVED'
    END AS match_quality,
    selected_candidate_count AS candidate_match_count,
    CASE
        WHEN match_status = 'AMBIGUOUS' AND target_claim_count > 1
            THEN CONCAT('PSGC place reached by ', target_claim_count, ' source rows through ', selected_match_method)
        WHEN match_status = 'AMBIGUOUS'
            THEN CONCAT('Multiple contextual candidates: method=', selected_match_method, ', candidates=', selected_candidate_count)
        WHEN match_status = 'UNMATCHED' AND sheet_candidate_count <> 1 THEN CONCAT('Sheet context candidate count=', sheet_candidate_count)
        WHEN match_status = 'UNMATCHED' AND locality_context_code IS NULL THEN 'No verified locality context'
        WHEN match_status = 'INVALID_SOURCE' THEN 'Missing mandatory Table C source identity or place name'
    END AS ambiguity_or_exception_reason,
    CASE
        WHEN population_count IS NULL OR population_count <= 0 THEN 'INVALID_POPULATION'
        WHEN match_status = 'INVALID_SOURCE' THEN 'SOURCE_EXCEPTION'
        WHEN match_status = 'AMBIGUOUS' THEN 'AMBIGUOUS'
        WHEN match_status = 'UNMATCHED' THEN 'UNMATCHED'
        WHEN matched_place_type = 'BARANGAY' THEN 'ACCEPTED_PRIMARY'
        ELSE 'MATCHED_NON_PRIMARY_LEVEL'
    END AS row_disposition,
    psgc_population_crosscheck,
    CASE
        WHEN population_count IS NOT NULL AND psgc_population_crosscheck IS NOT NULL
            THEN population_count - psgc_population_crosscheck
    END AS population_difference,
    CASE
        WHEN population_count > 0 AND psgc_population_crosscheck IS NOT NULL
            THEN ROUND(100.0 * (population_count - psgc_population_crosscheck) / population_count, 4)
    END AS population_difference_pct,
    population_reference_date,
    reference_year,
    source_name,
    source_snapshot_id,
    source_ingest_run_id,
    psgc_source_snapshot_id,
    psgc_source_ingest_run_id,
    psgc_version,
    alias_version,
    transformation_rule_version,
    SHA2(CONCAT_WS('|', source_snapshot_id, psgc_source_snapshot_id, alias_version, transformation_rule_version), 256) AS run_id,
    source_load_ts
FROM enriched;


# Summary

Created `02-silver.silver_population_place_reconciliation` at one result per Table C source row and Silver run. Its natural key retains source file, sheet, row number, and run ID. The table records the Table C and PSGC snapshots, both Bronze ingest identities, PSGC and alias versions, sheet context method, locality context, match method, final row disposition, population parsing, and diagnostic cross-check. Every Table C row stays visible. Requires Databricks execution. Next dependency: region population aggregation.


In [0]:
SELECT
    source_snapshot_id,
    psgc_source_snapshot_id,
    alias_version,
    match_status,
    row_disposition,
    COUNT(*) AS source_rows
FROM `02-silver`.silver_population_place_reconciliation
GROUP BY source_snapshot_id, psgc_source_snapshot_id, alias_version, match_status, row_disposition
ORDER BY row_disposition, match_status;
